# Manuscript Runtime Tables

Discounted, total-reward, MTG, JMLR, and manuscript table-generation recipes.

All cells are intentionally unexecuted; run sections selectively from top to bottom.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / 'core').exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
ARTIFACTS = ROOT / 'artifacts'
ARTIFACTS.mkdir(exist_ok=True)


## `exps/discounted_results_manuscrit.py`

Full discounted manuscript table.


In [ ]:
import os
import pandas as pd
from utils.data_management import solve, import_models_from_file
from core.model import GenericModel
from utils.paths import EXPS_ARTIFACTS_PATH

os.environ["KMP_DUPLICATE_LIB_OK"] = "True"
EXPS_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)

discount = 0.99
precision = 0.001
repeat = 5

solver_list = [
    "mdptoolbox_vi",
    "mdptoolbox_vigs",
    "mdpsolver_vi",
    # "mdpsolver_vigs",
    # "mdpsolver_visor",
    "marmote_vi",
    "marmote_vigs",
    "mdpsolver_pi",
    "mdptoolbox_pim",
    "mdpsolver_mpi",
    "mdpsolver_mpigs",
    # "mdpsolver_mpisor",
    "marmote_mpi",
    "marmote_mpigs",
    "gurobi_primal",
    "gurobi_dual",
]

combinations = [
    ("rooms", 100, 4),
    ("taxi", 2000, 6),
    ("barto", 2000, 10),
    ("cliff", 2000, 4),
    ("tandem", 2000, 9),
    ("dam", 2000, 10),
    ("inventory", 2000, 50),
    ("impatience", 500, 10),
    ("replacement", 500, 20),
    ("garnet", 500, 10),
    ("block", 1000, 10),
    ("maze_backtrack", 2000, 4),
    ("maze_prims", 2000, 4),
    ("maze_wilson", 2000, 4),
    ("swim", 5000, 2),
    ('unichain', 5000, 2),
    ("sutton", 2000, 20),
    ("mountain", 2000, 3),
    ("oil", 400, 50),
    ("ambulance", 600, 50),
    ("inventory", 1000, 200),
    ("replacement", 1000, 50),
    ("tandem_choice", 1024, 100),
    ("garnet", 500, 200),
    ("block", 800, 200),
]
row_order = ["Solver", "StateDim", "ActionDim", "Density"] + solver_list

# Initialize storage
all_results = {}

for model_name, state_dim, action_dim in combinations:
    model = import_models_from_file(model_name)
    model: GenericModel = model(state_dim, action_dim)
    model.create_model()

    density = model.get_transition_density()

    # Initialize dict for this model
    all_results[model_name] = {
        "Solver": " ",
        "StateDim": state_dim,
        "ActionDim": action_dim,
        "Density": f"{density:.2e}",
    }

    for solver_name in solver_list:
        if "gurobi" in solver_name and state_dim > 250:
            all_results[model_name][solver_name] = "---"
            continue
        model, solver, mean, std = solve(
            model_name,
            solver_name,
            state_dim=state_dim,
            action_dim=action_dim,
            discount=discount,
            precision=precision,
            repeat=repeat,
            verbose=True,
        )
        all_results[model_name][solver_name] = f"{mean:.2f} $\\pm$ {std:.2f}"

    df_temp = pd.DataFrame(all_results).loc[row_order]
    print()
    latex_code = df_temp.to_latex(escape=False)
    print(latex_code)
    print()
    # Save LaTeX code to a tex file
    with open(EXPS_ARTIFACTS_PATH / "results_discounted.tex", "w") as f:
        f.write(latex_code)
    df_temp.to_csv(EXPS_ARTIFACTS_PATH / "results_discounted.csv", index=False)

# Convert dict to DataFrame
df = pd.DataFrame(all_results)

# Reorder rows: metadata first, then solvers

df = df.loc[row_order]

# Print LaTeX table
print(df.to_latex(escape=False))


## `exps/discounted_results_manuscrit_real.py`

Discounted manuscript table for real/problem-like models.


In [ ]:
import os
import pandas as pd
from utils.data_management import solve, import_models_from_file
from core.model import GenericModel
from utils.paths import EXPS_ARTIFACTS_PATH

os.environ["KMP_DUPLICATE_LIB_OK"] = "True"
EXPS_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)

discount = 0.99
precision = 0.001
repeat = 5

solver_list = [
    "mdptoolbox_vi",
    "mdpsolver_vi",
    # "mdpsolver_vigs",
    # "mdpsolver_visor",
    "marmote_vi",
    "marmote_vigs",
    "mdpsolver_pi",
    "mdptoolbox_pim",
    "mdpsolver_mpi",
    "mdpsolver_mpigs",
    # "mdpsolver_mpisor",
    "marmote_mpi",
    "marmote_mpigs",
]

combinations = [
    ("tandem", 4000, 9),
    ("dam", 4000, 50),
    ("inventory", 4000, 50),
    ("impatience", 2000, 10),
    ("replacement", 1000, 50),
]
row_order = ["Solver", "StateDim", "ActionDim", "Density"] + solver_list

# Initialize storage
all_results = {}

for model_name, state_dim, action_dim in combinations:
    model = import_models_from_file(model_name)
    model: GenericModel = model(state_dim, action_dim)
    model.create_model()

    state_dim, action_dim = model.state_dim, model.action_dim

    density = model.get_transition_density()

    # Initialize dict for this model
    all_results[model_name] = {
        "Solver": " ",
        "StateDim": state_dim,
        "ActionDim": action_dim,
        "Density": f"{density:.2e}",
    }

    for solver_name in solver_list:
        model, solver, mean, std = solve(
            model_name,
            solver_name,
            state_dim=state_dim,
            action_dim=action_dim,
            discount=discount,
            precision=precision,
            repeat=repeat,
            verbose=True,
        )
        all_results[model_name][solver_name] = f"{mean:.2f} $\\pm$ {std:.2f}"

    df_temp = pd.DataFrame(all_results).loc[row_order]
    print()
    latex_code = df_temp.to_latex(escape=False)
    print(latex_code)
    print()
    # Save LaTeX code to a tex file
    with open(EXPS_ARTIFACTS_PATH / "results_discounted_real.tex", "w") as f:
        f.write(latex_code)

# Convert dict to DataFrame
df = pd.DataFrame(all_results)

# Reorder rows: metadata first, then solvers

df = df.loc[row_order]

# Print LaTeX table
print(df.to_latex(escape=False))


## `exps/discounted_results_manuscrit_rnd.py`

Discounted manuscript table for random models.


In [ ]:
import os
import pandas as pd
from utils.data_management import solve, import_models_from_file
from core.model import GenericModel
from utils.paths import EXPS_ARTIFACTS_PATH

os.environ["KMP_DUPLICATE_LIB_OK"] = "True"
EXPS_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)

discount = 0.99
precision = 0.001
repeat = 5

solver_list = [
    "mdptoolbox_vi",
    "mdpsolver_vi",
    # "mdpsolver_vigs",
    # "mdpsolver_visor",
    "marmote_vi",
    "marmote_vigs",
    "mdpsolver_pi",
    "mdptoolbox_pim",
    "mdpsolver_mpi",
    "mdpsolver_mpigs",
    # "mdpsolver_mpisor",
    "marmote_mpi",
    "marmote_mpigs",
    "gurobi_primal",
    "gurobi_dual",
]

combinations = [
    ("garnet", 500, 10),
    ("garnet", 500, 50),
    ("block", 500, 10),
    ("block", 500, 50),
]
row_order = ["Solver", "StateDim", "ActionDim", "Density"] + solver_list

# Initialize storage
all_results = {}

for model_name, state_dim, action_dim in combinations:
    model = import_models_from_file(model_name)
    model: GenericModel = model(state_dim, action_dim)
    model.create_model()

    density = model.get_transition_density()

    # Initialize dict for this model
    all_results[model_name] = {
        "Solver": " ",
        "StateDim": state_dim,
        "ActionDim": action_dim,
        "Density": f"{density:.2e}",
    }

    for solver_name in solver_list:
        if "gurobi" in solver_name and state_dim > 250:
            all_results[model_name][solver_name] = "---"
            continue
        model, solver, mean, std = solve(
            model_name,
            solver_name,
            state_dim=state_dim,
            action_dim=action_dim,
            discount=discount,
            precision=precision,
            repeat=repeat,
            verbose=True,
        )
        all_results[model_name][solver_name] = f"{mean:.2f} $\\pm$ {std:.2f}"

    df_temp = pd.DataFrame(all_results).loc[row_order]
    print()
    latex_code = df_temp.to_latex(escape=False)
    print(latex_code)
    print()
    # Save LaTeX code to a tex file
    with open(EXPS_ARTIFACTS_PATH / "results_discounted.tex", "w") as f:
        f.write(latex_code)
    df_temp.to_csv(EXPS_ARTIFACTS_PATH / "results_discounted.csv", index=False)

# Convert dict to DataFrame
df = pd.DataFrame(all_results)

# Reorder rows: metadata first, then solvers

df = df.loc[row_order]

# Print LaTeX table
print(df.to_latex(escape=False))


## `exps/discounted_results_manuscrit_toy.py`

Discounted manuscript table for toy models.


In [ ]:
import os
import pandas as pd
from utils.data_management import solve, import_models_from_file
from core.model import GenericModel
from utils.paths import EXPS_ARTIFACTS_PATH

os.environ["KMP_DUPLICATE_LIB_OK"] = "True"
EXPS_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)

discount = 0.99
precision = 0.001
repeat = 5

solver_list = [
    "mdptoolbox_vi",
    "mdpsolver_vi",
    # "mdpsolver_vigs",
    # "mdpsolver_visor",
    "marmote_vi",
    "marmote_vigs",
    "mdpsolver_pi",
    "mdptoolbox_pim",
    "mdpsolver_mpi",
    "mdpsolver_mpigs",
    # "mdpsolver_mpisor",
    "marmote_mpi",
    "marmote_mpigs",
]

combinations = [
    # ("rooms", 4000, 4),
    # ("taxi", 4000, 6),
    # ("barto", 4000, 10),
    # ("cliff", 4000, 4),
]
row_order = ["Solver", "StateDim", "ActionDim", "Density"] + solver_list

# Initialize storage
all_results = {}

for model_name, state_dim, action_dim in combinations:
    model = import_models_from_file(model_name)
    model: GenericModel = model(state_dim, action_dim)
    model.create_model()

    density = model.get_transition_density()

    # Initialize dict for this model
    all_results[model_name] = {
        "Solver": " ",
        "StateDim": model.state_dim,
        "ActionDim": model.action_dim,
        "Density": f"{density:.2e}",
    }

    for solver_name in solver_list:
        if "gurobi" in solver_name and state_dim > 500:
            all_results[model_name][solver_name] = "---"
            continue
        model, solver, mean, std = solve(
            model_name,
            solver_name,
            state_dim=state_dim,
            action_dim=action_dim,
            discount=discount,
            precision=precision,
            repeat=repeat,
            verbose=True,
        )
        all_results[model_name][solver_name] = f"{mean:.2f} $\\pm$ {std:.2f}"

    df_temp = pd.DataFrame(all_results).loc[row_order]
    print()
    latex_code = df_temp.to_latex(escape=False)
    print(latex_code)
    print()
    # Save LaTeX code to a tex file
    with open(EXPS_ARTIFACTS_PATH / "results_discounted_toy.tex", "w") as f:
        f.write(latex_code)
    df_temp.to_csv(EXPS_ARTIFACTS_PATH / "results_discounted_toy.csv", index=False)

# Convert dict to DataFrame
df = pd.DataFrame(all_results)

# Reorder rows: metadata first, then solvers

df = df.loc[row_order]

# Print LaTeX table
print(df.to_latex(escape=False))


## `exps/discounted_results_manuscrit_toy_2.py`

Second toy discounted manuscript table.


In [ ]:
import os
import pandas as pd
from utils.data_management import solve, import_models_from_file
from core.model import GenericModel
from utils.paths import EXPS_ARTIFACTS_PATH

os.environ["KMP_DUPLICATE_LIB_OK"] = "True"
EXPS_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)

discount = 0.99
precision = 0.001
repeat = 5

solver_list = [
    "mdptoolbox_vi",
    "mdpsolver_vi",
    # "mdpsolver_vigs",
    # "mdpsolver_visor",
    "marmote_vi",
    "marmote_vigs",
    "mdpsolver_pi",
    "mdptoolbox_pim",
    "mdpsolver_mpi",
    "mdpsolver_mpigs",
    # "mdpsolver_mpisor",
    "marmote_mpi",
    "marmote_mpigs",
]

combinations = [
    # ("maze_backtrack", 2000, 4),
    # ("maze_prims", 2000, 4),
    # ("maze_wilson", 2000, 4),
    # ("swim", 4000, 2),
    # ('unichain', 8000, 2),
    # ("sutton", 8000, 20),
    # ("mountain", 8000, 3),
    # ("oil", 600, 50),
    ("ambulance", 600, 50),
]
row_order = ["Solver", "StateDim", "ActionDim", "Density"] + solver_list

# Initialize storage
all_results = {}

for model_name, state_dim, action_dim in combinations:
    model = import_models_from_file(model_name)
    model: GenericModel = model(state_dim, action_dim)
    model.create_model()

    density = model.get_transition_density()

    # Initialize dict for this model
    all_results[model_name] = {
        "Solver": " ",
        "StateDim": model.state_dim,
        "ActionDim": model.action_dim,
        "Density": f"{density:.2e}",
    }

    for solver_name in solver_list:
        if "gurobi" in solver_name and state_dim > 250:
            all_results[model_name][solver_name] = "---"
            continue
        model, solver, mean, std = solve(
            model_name,
            solver_name,
            state_dim=state_dim,
            action_dim=action_dim,
            discount=discount,
            precision=precision,
            repeat=repeat,
            verbose=True,
        )
        all_results[model_name][solver_name] = f"{mean:.2f} $\\pm$ {std:.2f}"

    df_temp = pd.DataFrame(all_results).loc[row_order]
    print()
    latex_code = df_temp.to_latex(escape=False)
    print(latex_code)
    print()
    # Save LaTeX code to a tex file
    with open(EXPS_ARTIFACTS_PATH / "results_discounted_toy_2.tex", "w") as f:
        f.write(latex_code)
    df_temp.to_csv(EXPS_ARTIFACTS_PATH / "results_discounted_toy_2.csv", index=False)

# Convert dict to DataFrame
df = pd.DataFrame(all_results)

# Reorder rows: metadata first, then solvers

df = df.loc[row_order]

# Print LaTeX table
print(df.to_latex(escape=False))


## `exps/total_results_manuscrit.py`

Total-reward manuscript table.


In [ ]:
import os
import pandas as pd
from utils.data_management import solve, import_models_from_file
from core.model import GenericModel
from utils.paths import EXPS_ARTIFACTS_PATH

os.environ["KMP_DUPLICATE_LIB_OK"] = "True"
EXPS_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)

discount = 1.0
precision = 0.001
repeat = 5

solver_list = [
    "mdptoolbox_vi_total",
    # "mdpsolver_vigs",
    # "mdpsolver_visor",
    "marmote_vi_total",
    "marmote_vigs_total",
    "mdptoolbox_pim_total",
    # "mdpsolver_mpisor",
    "marmote_pim_total",
    "marmote_pimgs_total",
    # "gurobi_primal_total",
    # "gurobi_dual_total",
]

combinations = [
    # ("rooms_total", 100, 4),
    ("rooms_total", 1000, 4),
    # ("taxi_total", 800, 6),
    # ("barto_total", 1000, 20),
    # ("mountain_total", 4000, 3),
]
row_order = ["Solver", "StateDim", "ActionDim", "Density"] + solver_list

# Initialize storage
all_results = {}

for model_name, state_dim, action_dim in combinations:
    model = import_models_from_file(model_name)
    model: GenericModel = model(state_dim, action_dim)
    model.create_model()

    density = model.get_transition_density()

    # Initialize dict for this model
    all_results[model_name+str(state_dim)] = {
        "Solver": " ",
        "StateDim": state_dim,
        "ActionDim": action_dim,
        "Density": f"{density:.2e}",
    }

    for solver_name in solver_list:
        if "gurobi" in solver_name and state_dim > 250:
            all_results[model_name+str(state_dim)][solver_name] = "---"
            continue
        model, solver, mean, std = solve(
            model_name,
            solver_name,
            state_dim=state_dim,
            action_dim=action_dim,
            discount=discount,
            precision=precision,
            repeat=repeat,
            verbose=True,
        )
        all_results[model_name+str(state_dim)][solver_name] = f"{mean:.2f} $\\pm$ {std:.2f}"

    df_temp = pd.DataFrame(all_results).loc[row_order]
    print()
    latex_code = df_temp.to_latex(escape=False)
    print(latex_code)
    print()
    # Save LaTeX code to a tex file
    with open(EXPS_ARTIFACTS_PATH / "results_discounted.tex", "w") as f:
        f.write(latex_code)
    df_temp.to_csv(EXPS_ARTIFACTS_PATH / "results_discounted.csv", index=False)

# Convert dict to DataFrame
df = pd.DataFrame(all_results)

# Reorder rows: metadata first, then solvers

df = df.loc[row_order]

# Print LaTeX table
print(df.to_latex(escape=False))


## `exps/manuscrit_discounted.py`

Discounted manuscript run recipe.


In [ ]:
import os
from utils.data_management import solve, import_models_from_file
from core.model import GenericModel

os.environ["KMP_DUPLICATE_LIB_OK"] = "True"

discount = 0.99
precision = 0.001
repeat = 5

solver_list = [
    "mdptoolbox_vi",
    "mdptoolbox_vigs",
    "mdpsolver_vi",
    # "mdpsolver_vigs",
    # "mdpsolver_visor",
    "marmote_vi",
    "marmote_vigs",
    "mdpsolver_pi",
    "mdptoolbox_pim",
    "mdpsolver_mpi",
    "mdpsolver_mpigs",
    # "mdpsolver_mpisor",
    "marmote_mpi",
    "marmote_mpigs",
    "gurobi_primal",
    "gurobi_dual",
]

combinations = [
    ("rooms", 250, 4),
    ("taxi", 2500, 6),
    ("barto", 2500, 10),
    ("cliff", 2500, 4),
    ("tandem", 2500, 9),
    ("dam", 2500, 10),
    ("inventory", 2500, 50),
    ("impatience", 500, 10),
    ("garnet", 500, 10),
    ("block", 1000, 10),
    ("maze_backtrack", 2500, 4),
    ("maze_prims", 2500, 4),
    ("maze_wilson", 2500, 4),
    ("swim", 5000, 2),
    ("sutton", 2500, 20),
    ("mountain", 2500, 3),
    ("oil", 400, 50),
    ("ambulance", 600, 50),
    ("inventory", 1000, 200),
    ("replacement", 1000, 50),
    ("tandem_choice", 1024, 100),
    ("garnet", 500, 200),
    ("block", 800, 200),
]

for (
    model_name,
    state_dim,
    action_dim,
) in combinations:

    model = import_models_from_file(model_name)
    model: GenericModel = model(state_dim, action_dim)
    model.create_model()


    print(
        f"Model: {model_name}, Discount: {discount}, Precision: {precision}, Repeat: {repeat}, States: {model.state_dim}, Actions: {model.action_dim}, Density: {model.get_transition_density():2e}"
    )

    for solver_name in solver_list:
        model, solver, mean, std = solve(
            model_name,
            solver_name,
            state_dim=state_dim,
            action_dim=action_dim,
            discount=discount,
            precision=precision,
            repeat=repeat,
            verbose=True,
        )

    print()


## `exps/manuscrit_total.py`

Total-reward manuscript run recipe.


In [ ]:
from utils.data_management import solve, import_models_from_file

precision = 0.01
repeat = 3

import argparse

parser = argparse.ArgumentParser(description="Process some integers.")
parser.add_argument("--model", type=str, help="Model name")
parser.add_argument("--state", type=int, help="State dimension")
parser.add_argument("--action", type=int, help="Action dimension")

args = parser.parse_args()

print(f"Model: {args.model}")
print(f"Precision: {precision}")
print(f"Repeat: {repeat}")

model_name = args.model
state_dim = args.state
action_dim = args.action

discount = 1.0

model = import_models_from_file(model_name)
model = model(state_dim, action_dim)
model.create_model()
print(f"States: {model.state_dim}")
print(f"Actions: {model.action_dim}")

solver_list = [
    "mdptoolbox_vi_total",
    "marmote_vi_total",
    "marmote_vigs_total",
    "mdptoolbox_pim_total",
    "marmote_pim_total",
    "marmote_pimgs_total",
    # "mdptoolbox_pi",
    # "gurobi_primal",
    # "gurobi_dual",
]

for solver_name in solver_list:
    model, solver = solve(
        model_name,
        solver_name,
        state_dim=state_dim,
        action_dim=action_dim,
        discount=discount,
        precision=precision,
        repeat=repeat,
        verbose=True,
    )


## `exps/manuscrit_dis.py`

Subprocess launcher for discounted manuscript runs.


In [ ]:
import subprocess

# Define the set of (model, state) combinations
combinations = [
    # ("rooms", 250, 4),
    # ("taxi", 2500, 6),
    # ("barto", 2500, 10),
    # ("cliff", 2500, 4),
    # ("tandem", 2500, 9),
    # ("dam", 2500, 10),
    # ("inventory", 2500, 50),
    # ("impatience", 500, 10),
    # ("garnet", 500, 10),
    # ("block", 1000, 10),
    ("maze_backtrack", 2500, 4),
    ("maze_prims", 2500, 4),
    ("maze_wilson", 2500, 4),
    ("swim", 5000, 2),
    ("sutton", 2500, 20),
    ("mountain", 2500, 3),
    ("oil", 400, 50),
    ("ambulance", 600, 50),
    ("inventory", 1000, 200),
    ("replacement", 1000, 50),
    ("tandem_choice", 1024, 100),
    ("garnet", 500, 200),
    ("block", 800, 200),
]

# Generate and execute the commands
for model, state, action in combinations:
    command = f"python manuscrit_discounted.py --model {model} --state {state} --action {action}"
    print(f"Executing: {command}")
    subprocess.run(command, shell=True, check=True)


## `exps/mtg.py`

MTG discounted runtime line recipe.


In [ ]:
import numpy as np
import argparse
import os
from utils.data_management import import_solver_from_file, write_text, solve
from utils.paths import RESULTS_PATH

RESULTS_PATH.mkdir(parents=True, exist_ok=True)

parser = argparse.ArgumentParser()
parser.add_argument("--model", type=str)
parser.add_argument("--state", type=int)
parser.add_argument("--discount", type=float)
parser.add_argument("--precision", type=float)
args = parser.parse_args()

action_dim = 10

solver_list = [
    "marmote_vi",
    "marmote_vigs",
    "mdptoolbox_vi",
    "marmote_pim",
    "marmote_pimgs",
    "mdptoolbox_pim",
    "gurobi_primal",
    "gurobi_dual",
]

solver_name_list = [
    import_solver_from_file(solver_name).name for solver_name in solver_list
]
filename = os.path.join(
    RESULTS_PATH,
    "marmote_{}_{}_{}_{}.txt".format(args.model, args.state, action_dim, args.discount),
)

print(
    "Model : {}, Statedim : {}, Actiondim : {}, Discount : {}".format(
        args.model, args.state, action_dim, args.discount
    )
)
print()

headline = "$\statedim$ & " + " & ".join(solver_name_list) + " \\\\"
write_text(filename, headline, show=True)
latex_line = "{}".format(args.state)

for solver_name in solver_list:
    if "gurobi" in solver_name and args.state > 25000:
        continue
    model, solver = solve(
        args.model, solver_name, args.state, action_dim, args.discount, args.precision
    )
    latex_line += " & {}".format(np.round(solver.runtime, 1))
    write_text(filename, latex_line, show=True)

latex_line += " \\\\"
write_text(filename, latex_line, show=True)


## `exps/mtg_total.py`

MTG total-reward runtime line recipe.


In [ ]:
import numpy as np
import argparse
import os

from utils.data_management import import_solver_from_file
from utils.data_management import write_text, solve
from utils.paths import RESULTS_PATH

RESULTS_PATH.mkdir(parents=True, exist_ok=True)

parser = argparse.ArgumentParser()
parser.add_argument("--model", type=str)
parser.add_argument("--state", type=int)
parser.add_argument("--precision", type=float)
args = parser.parse_args()

action_dim = 10

solver_list = [
    # "marmote_vi_total",
    # "marmote_vigs_total",
    # "mdptoolbox_vi_total",
    # "marmote_pim_total",
    # "marmote_pimgs_total",
    # "mdptoolbox_pim_total",
    "gurobi_primal_total",
    "gurobi_dual_total",
]

solver_name_list = [
    import_solver_from_file(solver_name).name for solver_name in solver_list
]
filename = os.path.join(
    RESULTS_PATH,
    "marmote_{}_{}_{}.txt".format(args.model, args.state, action_dim),
)

print(
    "Model : {}, Statedim : {}, Actiondim : {}".format(
        args.model, args.state, action_dim
    )
)
print()

headline = "$\statedim$ & " + " & ".join(solver_name_list) + " \\\\"
write_text(filename, headline, show=True)
latex_line = "{}".format(args.state)

for solver_name in solver_list:
    if "gurobi" in solver_name and args.state > 25000:
        continue
    model, solver = solve(
        args.model, solver_name, args.state, action_dim, 1.0, args.precision
    )
    latex_line += " & {}".format(np.round(solver.runtime, 1))
    write_text(filename, latex_line, show=True)

latex_line += " \\\\"
write_text(filename, latex_line, show=True)


## `exps/solve_jmlr.py`

JMLR discounted solve recipe.


In [ ]:
from utils.data_management import solve
import argparse

# from reset import *


parser = argparse.ArgumentParser()
parser.add_argument("--model", type=str)
parser.add_argument("--state", type=int)
parser.add_argument("--discount", type=float)
parser.add_argument("--action", type=int, required=False)
parser.add_argument("--precision", type=float, required=False)
parser.add_argument("--repeat", type=int, required=False)
args = parser.parse_args()

import sys

import sys
from types import ModuleType, FunctionType
from gc import get_referents

# Custom objects know their class.
# Function objects seem to know way too much, including modules.
# Exclude modules as well.
BLACKLIST = type, ModuleType, FunctionType


def getsize(obj):
    """sum size of object & members."""
    if isinstance(obj, BLACKLIST):
        raise TypeError("getsize() does not take argument of type: " + str(type(obj)))
    seen_ids = set()
    size = 0
    objects = [obj]
    while objects:
        need_referents = []
        for obj in objects:
            if not isinstance(obj, BLACKLIST) and id(obj) not in seen_ids:
                seen_ids.add(id(obj))
                size += sys.getsizeof(obj)
                need_referents.append(obj)
        objects = get_referents(*need_referents)
    return size


print(f"Model {args.model}...")

for solver in [
    # "aggregated_vi",
    # "aggregated_qvi",
    # "personal_vi",
    "aggregated_pim",
    # "personal_pim",
    # "bertsekas_pi",
]:
    if 'agg' not in solver:
        continue
    print("Solver:", solver)
    args.solver = solver

    assert args.model is not None, "Model must be specified."
    assert args.solver is not None, "Solver must be specified."
    assert args.state is not None, "State dim must be specified. --state <int>"
    assert (
        args.discount is not None
    ), "Discount factor must be specified., --discount <float>"
    args.action = 10 if args.action is None else args.action
    args.precision = 1e-1 if args.precision is None else args.precision
    args.repeat = 1 if args.repeat is None else args.repeat

    model, solver = solve(
        args.model,
        args.solver,
        args.state,
        args.action,
        args.discount,
        args.precision,
        args.repeat,
    )
    # print(f"Runtime: {solver.runtime}")
    # print(solver.value)

    if hasattr(solver, "partition"):
        print(f"Regions: {solver.partition.n_regions} / {model.state_dim}")


## `exps/solve_jmlr_total.py`

JMLR total-reward solve recipe.


In [ ]:
from utils.data_management import solve
import argparse

# from reset import *

parser = argparse.ArgumentParser()
parser.add_argument("--model", type=str)
parser.add_argument("--state", type=int)
parser.add_argument("--action", type=int, required=False)
parser.add_argument("--precision", type=float, required=False)
parser.add_argument("--repeat", type=int, required=False)
args = parser.parse_args()

print(f"Model {args.model}...")

for solver in [
    "aggregated_vi_total",
    "aggregated_qvi_total",
    "personal_vi",
    "aggregated_pim_total",
    "personal_pim",
]:
    args.solver = solver

    assert args.model is not None, "Model must be specified."
    assert args.solver is not None, "Solver must be specified."
    assert args.state is not None, "State dim must be specified. --state <int>"
    args.action = 10 if args.action is None else args.action
    args.precision = 1e-2 if args.precision is None else args.precision
    args.repeat = 1 if args.repeat is None else args.repeat

    model, solver = solve(
        args.model,
        args.solver,
        args.state,
        args.action,
        1.0,
        args.precision,
        args.repeat,
    )
    # print(f"Runtime: {solver.runtime}")
    # print(solver.value)

    # if hasattr(solver, "partition"):
    #     print(f"Regions: {solver.partition.n_regions}")


## `notebooks/thesis/plot_manuscrit.ipynb`

Thesis/manuscript plotting notebook.


In [ ]:
# Code for Rooms and Marmote VI vs Marmote MPI

from utils.data_management import solve

solver_list = ['marmote_vi', 'marmote_mpi']
model_name = "rooms"

state_dims = [500, 1000, 2000, 4000, 8000, 16000, 32000, 64000, 128000, 256000, 512000]
action_dim = 4
discount = 0.99
precision = 1e-2
repeat = 4

runtimes_vi, stds_vi = [], []
for state_dim in state_dims:
    model, solver, mean, std = solve(model_name, solver_list[0], state_dim, action_dim, discount, precision, repeat)
    print(state_dim, mean, std)
    runtimes_vi.append(mean)
    stds_vi.append(std)

runtimes_mpi, stds_mpi = [], []
for state_dim in state_dims:
    model, solver, mean, std = solve(model_name, solver_list[1], state_dim, action_dim, discount, precision, repeat)
    print(state_dim, mean, std)
    runtimes_mpi.append(mean)
    stds_mpi.append(std)

import matplotlib.pyplot as plt

FILENAME = r"C:\Users\orsof\Manuscrit\Chapter5MDP\images\vi_vs_mpi_marmote.pdf"

plt.errorbar(state_dims, runtimes_vi, yerr=stds_vi, label='Marmote VI', marker='o')
plt.errorbar(state_dims, runtimes_mpi, yerr=stds_mpi, label='Marmote MPI', marker='o')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('State Space Size', fontsize=18)
plt.ylabel('Runtime (s)', fontsize=18)
plt.legend(fontsize=18)
plt.tight_layout()
plt.savefig(FILENAME)
plt.show()


In [ ]:
# Code for Rooms and Marmote VI vs Marmote MPI

from utils.data_management import solve

solver_list = ['marmote_vi', 'marmote_mpi']
model_name = "rooms"

state_dims = [500, 1000, 2000, 4000, 8000, 16000, 32000, 64000, 128000, 256000]
action_dim = 4
discount = 0.999
precision = 1e-2
repeat = 4

runtimes_vi, stds_vi = [], []
for state_dim in state_dims:
    model, solver, mean, std = solve(model_name, solver_list[0], state_dim, action_dim, discount, precision, repeat)
    print(state_dim, mean, std)
    runtimes_vi.append(mean)
    stds_vi.append(std)

runtimes_mpi, stds_mpi = [], []
for state_dim in state_dims:
    model, solver, mean, std = solve(model_name, solver_list[1], state_dim, action_dim, discount, precision, repeat)
    print(state_dim, mean, std)
    runtimes_mpi.append(mean)
    stds_mpi.append(std)

import matplotlib.pyplot as plt

FILENAME = r"C:\Users\orsof\Manuscrit\Chapter5MDP\images\vi_vs_mpi_marmote_999.pdf"

plt.errorbar(state_dims, runtimes_vi, yerr=stds_vi, label='Marmote VI', marker='o')
plt.errorbar(state_dims, runtimes_mpi, yerr=stds_mpi, label='Marmote MPI', marker='o')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('State Space Size', fontsize=18)
plt.ylabel('Runtime (s)', fontsize=18)
plt.legend(fontsize=18)
plt.tight_layout()
plt.savefig(FILENAME)
plt.show()


In [ ]:
import os
import pandas as pd
from utils.data_management import solve, import_models_from_file
from utils.generic_model import GenericModel

os.environ["KMP_DUPLICATE_LIB_OK"] = "True"

discount = 0.99
precision = 0.001
repeat = 5

solver_list = [
    "mdptoolbox_vi",
    "mdptoolbox_vigs",
    "mdpsolver_vi",
    # "mdpsolver_vigs",
    # "mdpsolver_visor",
    "marmote_vi",
    "marmote_vigs",
    "mdptoolbox_pim",
    "mdpsolver_mpi",
    "mdpsolver_mpigs",
    # "mdpsolver_mpisor",
    "marmote_mpi",
    "marmote_mpigs",
    "gurobi_primal",
    "gurobi_dual",
]

state_dim = 2000
action_dim = 10

combinations = [
    ("rooms", state_dim, action_dim),
    ("taxi", state_dim, action_dim),
    ("barto", state_dim, action_dim),
    ("cliff", state_dim, action_dim),
    ("tandem", state_dim, action_dim),
    ("dam", state_dim, action_dim),
    ("inventory", state_dim, action_dim),
    ("garnet", state_dim, action_dim),
    ("block", state_dim, action_dim),
    ("maze_backtrack", state_dim, action_dim),
    ("maze_prims", state_dim, action_dim),
    ("maze_wilson", state_dim, action_dim),
    ("swim", state_dim, action_dim),
    ("unichain", state_dim, action_dim),
    ("sutton", state_dim, action_dim),
    ("mountain", state_dim, action_dim),
    ("oil", state_dim, action_dim),
    ("ambulance", state_dim, action_dim),
    ("inventory", state_dim, action_dim),
    ("replacement", state_dim, action_dim),
    ("tandem_choice", state_dim, action_dim),
]

vi_mean_runtimes = []
vi_std_runtimes = []

mpi_mean_runtimes = []
mpi_std_runtimes = []

for model_name, state_dim, action_dim in combinations:
    model = import_models_from_file(model_name)
    model: GenericModel = model(state_dim, action_dim)
    model.create_model()

    density = model.get_transition_density()

    model, solver, mean, std = solve(
        model_name,
        'marmote_vi',
        state_dim=state_dim,
        action_dim=action_dim,
        discount=discount,
        precision=precision,
        repeat=repeat,
        verbose=True,
    )

    vi_mean_runtimes.append((density, mean))
    vi_std_runtimes.append((density, std))

    model, solver, mean, std = solve(
        model_name,
        'marmote_mpi',
        state_dim=state_dim,
        action_dim=action_dim,
        discount=discount,
        precision=precision,
        repeat=repeat,
        verbose=True,
    )

    mpi_mean_runtimes.append((density, mean))
    mpi_std_runtimes.append((density, std))

vi_mean_runtimes.sort(key=lambda x: x[0])
vi_std_runtimes.sort(key=lambda x: x[0])
mpi_mean_runtimes.sort(key=lambda x: x[0])
mpi_std_runtimes.sort(key=lambda x: x[0])


import matplotlib.pyplot as plt
import numpy as np

plt.errorbar(
    [x[0] for x in vi_mean_runtimes],
    [x[1] for x in vi_mean_runtimes],
    yerr=[x[1] for x in vi_std_runtimes],
    label="Marmote VI",
    marker="o",
)

plt.errorbar(
    [x[0] for x in mpi_mean_runtimes],
    [x[1] for x in mpi_mean_runtimes],
    yerr=[x[1] for x in mpi_std_runtimes],
    label="Marmote MPI",
    marker="o",
)

plt.xscale("log")
plt.yscale("log")
plt.xlabel("Transition Density", fontsize=18)
plt.ylabel("Runtime (s)", fontsize=18)
plt.legend(fontsize=18)
plt.tight_layout()
plt.savefig(r"C:\Users\orsof\Manuscrit\Chapter5MDP\images\vi_vs_mpi_marmote_density.pdf")
plt.show()


In [ ]:
for model_name, state_dim, action_dim in combinations:
    model = import_models_from_file(model_name)
    model: GenericModel = model(state_dim, action_dim)
    model.create_model()

    normalized_runtime_vi = [x[1] / model.action_dim for x in vi_mean_runtimes]
    normalized_runtime_mpi = [x[1] / model.action_dim for x in mpi_mean_runtimes]

densities = [x[0] for x in vi_mean_runtimes]

plt.cla()

plt.plot(densities, normalized_runtime_vi, label="Marmote VI")
plt.plot(densities, normalized_runtime_mpi, label="Marmote MPI")

plt.xscale("log")
plt.yscale("log")
plt.xlabel("Transition Density", fontsize=18)
plt.ylabel("Runtime (s)", fontsize=18)
plt.legend(fontsize=18)
plt.tight_layout()
plt.savefig(r"C:\Users\orsof\Manuscrit\Chapter5MDP\images\vi_vs_mpi_marmote_density.pdf")
plt.show()


### Comparison of building times


In [ ]:
start, end, steps = 1000, 10000, 8

states = (10 ** np.linspace(np.log10(start), np.log10(end), steps)).astype(int)

from tqdm import trange, tqdm

n_exp = 3

mdpt = []
marmote = []
mdps = []

for state in tqdm(states):
    runtimes_mdpt = [test_building_sparse(state) for _ in range(n_exp)]
    mean, std = np.mean(runtimes_mdpt), np.std(runtimes_mdpt)
    mdpt.append((mean, std))

    runtimes_marmote = [test_building_marmote(state) for _ in range(n_exp)]
    mean, std = np.mean(runtimes_marmote), np.std(runtimes_marmote)
    marmote.append((mean, std))

    runtimes_mdps = [test_building_mdps(state) for _ in range(n_exp)]
    mean, std = np.mean(runtimes_mdps), np.std(runtimes_mdps)
    mdps.append((mean, std))

import matplotlib.pyplot as plt
import numpy as np
import os

FILENAME = r"C:\Users\orsof\Manuscrit\Chapter5MDP\images\building_times_rooms.pdf"

plt.errorbar(
    states,
    [x[0] for x in mdpt],
    yerr=[x[1] for x in mdpt],
    label="MDPToolbox Sparse",
    marker="o",
)
plt.errorbar(
    states,
    [x[0] for x in marmote],
    yerr=[x[1] for x in marmote],
    label="Marmote Sparse",
    marker="o",
)
plt.errorbar(
    states,
    [x[0] for x in mdps],
    yerr=[x[1] for x in mdps],
    label="MDPSolver Sparse",
    marker="o",
)
plt.xscale("log")
plt.yscale("log")
plt.xlabel("State Space Size", fontsize=18)
plt.ylabel("Building Time (s)", fontsize=18)
plt.legend(fontsize=18)
plt.tight_layout()
plt.savefig(FILENAME)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

FILENAME = r"C:\Users\orsof\Manuscrit\Chapter5MDP\images\building_times_rooms.pdf"

states = states[:-1]
mdpt = mdpt[:-1]
marmote = marmote[:-1]
mdps = mdps[:-1]

plt.errorbar(
    states,
    [x[0] for x in mdpt],
    yerr=[x[1] for x in mdpt],
    label="MDPToolbox Sparse",
    marker="o",
)
plt.errorbar(
    states,
    [x[0] for x in marmote],
    yerr=[x[1] for x in marmote],
    label="Marmote Sparse",
    marker="o",
)
plt.errorbar(
    states,
    [x[0]*10 for x in mdps],
    yerr=[x[1] for x in mdps],
    label="MDPSolver Sparse",
    marker="o",
)
plt.xscale("log")
plt.yscale("log")
plt.xlabel("State Space Size", fontsize=18)
plt.ylabel("Building Time (s)", fontsize=18)
plt.legend(fontsize=18)
plt.tight_layout()
plt.savefig(FILENAME)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

FILENAME = r"C:\Users\orsof\Manuscrit\Chapter5MDP\images\building_times_rooms.pdf"

states = states[1:-1]
mdpt = mdpt[1:-1]
marmote = marmote[1:-1]
mdps = mdps[1:-1]

val = marmote[0][0]

plt.errorbar(
    states,
    [x[0] for x in mdpt],
    yerr=[x[1] for x in mdpt],
    label="MDPToolbox Sparse",
    marker="o",
)
plt.errorbar(
    states,
    [x[0] for x in marmote],
    yerr=[x[1] for x in marmote],
    label="Marmote Sparse",
    marker="o",
)
plt.errorbar(
    states,
    [x[0]/val for x in mdps],
    yerr=[x[1] for x in mdps],
    label="MDPSolver Sparse",
    marker="o",
)
plt.xscale("log")
plt.yscale("log")
plt.xlabel("State Space Size", fontsize=18)
plt.ylabel("Building Time (s)", fontsize=18)
plt.legend(fontsize=18)
plt.tight_layout()
plt.savefig(FILENAME)
plt.show()
